# CMIP-derived ACC perturbations: GenCast vs IFS

This notebook implements the paper-style ACC workflow for both GenCast and IFS using one shared CMIP-derived perturbation source:

`ACC_delta = mean(1980-2014) - mean(1850-1900)`

The CMIP multi-model-mean ACC deltas are computed once, then interpolated to each forecast system's initialization date and target grid. Counterfactual initial conditions are built by subtracting those deltas from factual initial conditions. Forecast diagnostics then compare factual and counterfactual lagged-ensemble means for GenCast and IFS.

The notebook is intentionally configuration-driven. All paths, variable mappings, events, target grids, pressure levels, and output locations are editable in the configuration cell below.


## 1. Configuration

Edit this cell first. The defaults capture the file-layout conventions observed in `SSTDelta_vis.ipynb`, `CDFs_CRPScalc.ipynb`, and `Return_time_plots.ipynb`, but CMIP paths and counterfactual forecast paths are deliberately placeholders until you point them at the actual data/products.


In [ ]:
from pathlib import Path
import glob
import json
import math
import re
import warnings
from typing import Iterable

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cmip_acc_gencast_ifs_compare"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "models": ["GenCast", "IFS"],

    # Event/init-date list. These are the Pacific Northwest heat-event dates used
    # in the reference notebooks. Edit or extend as needed.
    "events": [
        {
            "event_id": "pnw_heat_2021_init_20210626",
            "init_date": "2021-06-26T00:00:00",
            "lead_days": 10,
            "event_window": ("2021-06-29T00:00:00", "2021-06-29T23:59:59"),
            "summary_region": {"latitude": (45.0, 52.0), "longitude": (-123.0, -119.0)},
        },
        {
            "event_id": "pnw_heat_2021_init_20210622",
            "init_date": "2021-06-22T00:00:00",
            "lead_days": 18,
            "event_window": ("2021-06-29T00:00:00", "2021-06-29T23:59:59"),
            "summary_region": {"latitude": (45.0, 52.0), "longitude": (-123.0, -119.0)},
        },
        {
            "event_id": "pnw_heat_2021_init_20210618",
            "init_date": "2021-06-18T00:00:00",
            "lead_days": 26,
            "event_window": ("2021-06-29T00:00:00", "2021-06-29T23:59:59"),
            "summary_region": {"latitude": (45.0, 52.0), "longitude": (-123.0, -119.0)},
        },
    ],

    "ensemble_size": 50,
    "lead_time_window_hours": (0, None),  # `(start, end)` relative to init; `None` means keep available lead range.

    # Variables to build from CMIP. Keep this list to variables required for the IC perturbations.
    # Use the same complete CMIP model set across these variables.
    "cmip_required_variables": ["tas", "ta", "hus", "ua", "va"],

    # Optional variables: loaded if available but not required for selecting the complete CMIP model set.
    # `zg` is optional because geopotential is normally adjusted hydrostatically from T/q perturbations.
    "cmip_optional_variables": ["zg", "ps", "psl"],

    # Variable mapping is explicit by design. If names differ in your data, edit these maps.
    "cmip_variable_map": {
        "tas": "tas",
        "ta": "ta",
        "hus": "hus",
        "ua": "ua",
        "va": "va",
        "zg": "zg",
        "ps": "ps",
        "psl": "psl",
    },

    "cmip": {
        # Option A: point this at a CSV with columns: source_id, variable_id, path.
        # `path` may be a concrete file, a glob, or a semicolon-separated list.
        "manifest_csv": None,

        # Option B: fill this explicit manifest list directly.
        "manifest": [
            # {"source_id": "MODEL_NAME", "variable_id": "tas", "path": "/path/to/tas_*.nc"},
        ],

        # Option C: provide source_ids and per-variable templates. Templates are only expanded
        # under these explicit roots; the notebook does not crawl the repository.
        "source_ids": [],
        "path_templates": {
            "tas": "/network/group/aopp/met_data/MET004_CMIP6/data/CCCma/CanESM5/historical/r1i1p1f1/Amon/tas/tas_Amon_CanESM5_historical_r1i1p1f1_gn_185001-201412.nc",
            "ta":  "/network/group/aopp/met_data/MET004_CMIP6/data/CCCma/CanESM5/historical/r1i1p1f1/Amon/ta/ta_Amon_CanESM5_historical_r1i1p1f1_gn_185001-201412.nc",
            # "hus": "/path/to/CMIP6/historical/{source_id}/Amon/hus/*.nc",
            "ua":  "/network/group/aopp/met_data/MET004_CMIP6/data/CCCma/CanESM5/historical/r1i1p1f1/Amon/ua/ua_Amon_CanESM5_historical_r1i1p1f1_gn_185001-201412.nc",
            "va":  "/network/group/aopp/met_data/MET004_CMIP6/data/CCCma/CanESM5/historical/r1i1p1f1/Amon/va/va_Amon_CanESM5_historical_r1i1p1f1_gn_185001-201412.nc",
        },

        "chunks": {"time": 120},
        "modern_period": (1980, 2014),
        "baseline_period": (1850, 1900),
    },

    "target_grid": {
        # Leave lat/lon as None to infer from each model's factual IC or factual forecast metadata.
        # Set explicit arrays/lists here if you want a fixed target grid.
        "latitude": None,
        "longitude": None,
        "pressure_levels_hpa": [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50],
        "horizontal_method": "linear",
        "pressure_method": "linear",
        "pressure_coordinate": "log",  # "log" or "linear"
    },

    "forecast_systems": {
        "GenCast": {
            "format": "zarr",
            "time_is_relative_to_init": True,
            "open_kwargs": {"consolidated": False},
            "target_grid_source": "factual_forecast",  # factual_ic or factual_forecast
            "requires_hydrostatic_geopotential": True,
            "geopotential_kind": "geopotential",  # geopotential => m2 s-2; height => m
            "hydrostatic_anchor": "top",

            # The seed templates preserve the convention from the existing notebooks.
            "factual_seed_templates": [
                "/network/group/aopp/predict/MK004_ANDRIYCHENKO_GENCAST/predictions/init_{init:%Y-%m-%dT%H}_{lead_days}d_26ens_0seed_predictions.zarr",
                "/network/group/aopp/predict/MK004_ANDRIYCHENKO_GENCAST/predictions/init_{init:%Y-%m-%dT%H}_{lead_days}d_24ens_1seed_predictions.zarr",
            ],
            "factual_template": "/network/group/aopp/predict/MK004_ANDRIYCHENKO_GENCAST/predictions/init_{init:%Y-%m-%dT%H}_{lead_days}d_50ens_26-0s_24-1s_predictions.zarr",

            # Point these at forecast products generated from the CMIP-ACC counterfactual ICs.
            "counterfactual_seed_templates": [
                str(OUTPUT_DIR / "forecasts/GenCast/cmip_acc_init_{init:%Y-%m-%dT%H}_{lead_days}d_26ens_0seed_predictions.zarr"),
                str(OUTPUT_DIR / "forecasts/GenCast/cmip_acc_init_{init:%Y-%m-%dT%H}_{lead_days}d_24ens_1seed_predictions.zarr"),
            ],
            "counterfactual_template": str(OUTPUT_DIR / "forecasts/GenCast/cmip_acc_init_{init:%Y-%m-%dT%H}_{lead_days}d_50ens_predictions.zarr"),

            # Set this to the factual GenCast initialization file used by your forecast-generation pipeline.
            "ic_template": "init_files/init_{init:%Y-%m-%dT%H}_{lead_days}d.nc",
            "counterfactual_ic_template": str(OUTPUT_DIR / "counterfactual_ics/GenCast/init_{init:%Y-%m-%dT%H}_{lead_days}d_cmip_acc.nc"),

            "variable_map": {
                "tas": "2m_temperature",
                "ta": "temperature",
                "hus": "specific_humidity",
                "ua": "u_component_of_wind",
                "va": "v_component_of_wind",
                "zg": "geopotential",
                "ps": "surface_pressure",
                "psl": "mean_sea_level_pressure",
            },
            "diagnostic_map": {
                "t2m": "2m_temperature",
            },
            "hydrostatic_variables": {
                "temperature": "temperature",
                "specific_humidity": "specific_humidity",
                "geopotential": "geopotential",
                "surface_geopotential": "geopotential_at_surface",
            },
        },

        "IFS": {
            "format": "netcdf",
            "time_is_relative_to_init": False,
            "open_kwargs": {"decode_timedelta": True},
            "target_grid_source": "factual_forecast",
            "requires_hydrostatic_geopotential": True,
            "geopotential_kind": "geopotential",
            "hydrostatic_anchor": "top",

            # Current/operational IFS convention observed in the reference notebooks.
            "factual_template": "/network/group/aopp/predict/MK004_ANDRIYCHENKO_GENCAST/control/oper_pf_{init:%Y-%m-%d}.nc",

            # Point this at IFS forecasts generated from the CMIP-ACC counterfactual ICs.
            "counterfactual_template": str(OUTPUT_DIR / "forecasts/IFS/cmip_acc_oper_pf_{init:%Y-%m-%d}.nc"),

            # Set this to the factual IFS initialization file used by your pipeline.
            "ic_template": "/path/to/ifs/factual_ic_{init:%Y-%m-%dT%H}.nc",
            "counterfactual_ic_template": str(OUTPUT_DIR / "counterfactual_ics/IFS/factual_ic_{init:%Y-%m-%dT%H}_cmip_acc.nc"),

            "variable_map": {
                "tas": "t2m",
                "ta": "t",
                "hus": "q",
                "ua": "u",
                "va": "v",
                "zg": "z",
                "ps": "sp",
                "psl": "msl",
            },
            "diagnostic_map": {
                "t2m": "t2m",
                "mx2t": "mx2t",
            },
            "hydrostatic_variables": {
                "temperature": "t",
                "specific_humidity": "q",
                "geopotential": "z",
                "surface_geopotential": "z_sfc",
            },
        },
    },

    "diagnostics": {
        "t2m": {
            "label": "2 m temperature",
            "units": "K",
            "summary_reducer": "mean_then_max_time",
        },
        # Optional if present in IFS files.
        "mx2t": {
            "label": "Daily maximum 2 m temperature",
            "units": "K",
            "summary_reducer": "mean_then_max_time",
        },
    },

    "outputs": {
        "cmip_delta_monthly_common": str(OUTPUT_DIR / "cmip_acc_delta_monthly_common.nc"),
        "cmip_delta_by_model": str(OUTPUT_DIR / "cmip_acc_delta_by_cmip_model.nc"),
        "interpolated_delta_template": str(OUTPUT_DIR / "interpolated_deltas/{model}/{event_id}_delta_on_target_grid.nc"),
        "result_template": str(OUTPUT_DIR / "results/{model}/{event_id}_acc_signal.nc"),
        "summary_csv": str(OUTPUT_DIR / "event_level_acc_summary.csv"),
        "plot_dir": str(OUTPUT_DIR / "plots"),
    },
}

pd.DataFrame(CONFIG["events"])


,event_id,init_date,lead_days,event_window,summary_region
0,pnw_heat_2021_init_20210626,2021-06-26T00:00:00,10,"(2021-06-29T00:00:00, 2021-06-29T23:59:59)","{'latitude': (45.0, 52.0), 'longitude': (-123...."
1,pnw_heat_2021_init_20210622,2021-06-22T00:00:00,18,"(2021-06-29T00:00:00, 2021-06-29T23:59:59)","{'latitude': (45.0, 52.0), 'longitude': (-123...."
2,pnw_heat_2021_init_20210618,2021-06-18T00:00:00,26,"(2021-06-29T00:00:00, 2021-06-29T23:59:59)","{'latitude': (45.0, 52.0), 'longitude': (-123...."


## 2. Load existing workflow conventions

Only the three requested notebooks were inspected. The conventions below are encoded in the loader helpers:

- GenCast predictions are zarr stores, often split by seed and concatenated along `sample`.
- Existing code renames `sample -> number`, `lat -> latitude`, and `lon -> longitude`.
- GenCast `time` is relative to initialization time, so diagnostics use `valid_time = init_date + time`.
- IFS/Nick-style NetCDF files use variables like `t2m`, `mx2t`, and often already use absolute `time`.
- Plotting style is plain Matplotlib with optional Cartopy maps and region/time summaries for the PNW heat-event box.


In [2]:
REFERENCE_NOTEBOOKS = [
    "SSTDelta_vis.ipynb",
    "CDFs_CRPScalc.ipynb",
    "Return_time_plots.ipynb",
]

REFERENCE_CONVENTIONS = {
    "GenCast": {
        "store": "zarr",
        "ensemble_dim_in_files": "sample",
        "standard_ensemble_dim": "number",
        "time_handling": "valid_time = init_date + relative time",
        "example_variable": "2m_temperature",
    },
    "IFS": {
        "store": "netcdf",
        "standard_ensemble_dim": "number",
        "time_handling": "time is absolute valid time",
        "example_variables": ["t2m", "mx2t"],
    },
    "coords": {
        "standard_lat": "latitude",
        "standard_lon": "longitude",
        "longitude_range": "normalized to [-180, 180] before regional selection/alignment",
    },
}

REFERENCE_CONVENTIONS


{'GenCast': {'store': 'zarr',
  'ensemble_dim_in_files': 'sample',
  'standard_ensemble_dim': 'number',
  'time_handling': 'valid_time = init_date + relative time',
  'example_variable': '2m_temperature'},
 'IFS': {'store': 'netcdf',
  'standard_ensemble_dim': 'number',
  'time_handling': 'time is absolute valid time',
  'example_variables': ['t2m', 'mx2t']},
 'coords': {'standard_lat': 'latitude',
  'standard_lon': 'longitude',
  'longitude_range': 'normalized to [-180, 180] before regional selection/alignment'}}

## Helper functions

These helpers keep the workflow explicit and reusable. They avoid broad repository/file discovery: all file opening uses configured paths, configured manifests, or configured glob templates.


In [3]:
COORD_ALIASES = {
    "latitude": ["latitude", "lat", "LATITUDE", "nav_lat"],
    "longitude": ["longitude", "lon", "LONGITUDE", "nav_lon"],
    "pressure": ["pressure", "plev", "level", "isobaricInhPa", "isobaricInPa"],
    "time": ["time", "valid_time"],
}

ENSEMBLE_ALIASES = ["number", "sample", "member", "realization", "ensemble"]
RD = 287.05
GRAVITY = 9.80665


def _is_path_placeholder(path: str) -> bool:
    return str(path).startswith("/path/to/") or str(path).strip() in {"", "None"}


def format_template(template, event, extra=None):
    """Format a configured path template with event fields and init-date variants."""
    init = pd.Timestamp(event["init_date"])
    values = dict(event)
    values.update(
        init=init.to_pydatetime(),
        init_ts=init,
        init_date=init.strftime("%Y-%m-%d"),
        init_hour=init.strftime("%H"),
        init_compact=init.strftime("%Y%m%d%H"),
    )
    if extra:
        values.update(extra)
    return str(template).format(**values)


def ensure_parent(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def standardize_coords(obj):
    """Rename common coordinate/dimension aliases and normalize lon/pressure units."""
    rename = {}
    names = set(obj.dims) | set(obj.coords)
    for standard, aliases in COORD_ALIASES.items():
        for alias in aliases:
            if alias in names and alias != standard and standard not in names:
                rename[alias] = standard
                break
    if rename:
        obj = obj.rename(rename)

    # Standardize ensemble dimension separately so `number` wins over other aliases.
    names = set(obj.dims) | set(obj.coords)
    if "number" not in names:
        for alias in ENSEMBLE_ALIASES:
            if alias in names and alias != "number":
                obj = obj.rename({alias: "number"})
                break

    if "longitude" in obj.coords:
        lon = obj["longitude"]
        try:
            if float(lon.max()) > 180.0:
                obj = obj.assign_coords(longitude=((lon + 180.0) % 360.0) - 180.0)
            obj = obj.sortby("longitude")
        except Exception as exc:
            warnings.warn(f"Could not normalize longitude coordinate: {exc}")

    if "latitude" in obj.coords:
        try:
            obj = obj.sortby("latitude")
        except Exception as exc:
            warnings.warn(f"Could not sort latitude coordinate: {exc}")

    if "pressure" in obj.coords:
        p = obj["pressure"]
        try:
            pmax = float(p.max())
            # Forecast files often use hPa levels; CMIP plev is normally Pa.
            if pmax < 2000.0:
                obj = obj.assign_coords(pressure=p * 100.0)
            obj["pressure"].attrs["units"] = "Pa"
        except Exception as exc:
            warnings.warn(f"Could not standardize pressure coordinate: {exc}")

    return obj


def squeeze_trivial_batch(ds):
    if "batch" in ds.dims and ds.sizes["batch"] == 1:
        return ds.squeeze("batch", drop=True)
    return ds


def subset_existing_variables(ds, variables, context="dataset"):
    if not variables:
        return ds
    variables = list(dict.fromkeys([v for v in variables if v]))
    present = [v for v in variables if v in ds.data_vars]
    missing = [v for v in variables if v not in ds.data_vars]
    if missing:
        warnings.warn(f"{context}: missing requested variables: {missing}")
    if not present:
        warnings.warn(f"{context}: none of the requested variables are present; returning coordinates only.")
        return ds[[]]
    return ds[present]


def open_configured_dataset(path, file_format=None, open_kwargs=None, variables=None, context="dataset"):
    """Open one configured dataset/store and subset requested variables by name."""
    path = str(path)
    open_kwargs = dict(open_kwargs or {})
    if _is_path_placeholder(path):
        raise FileNotFoundError(f"{context}: placeholder path is not configured: {path}")
    if not Path(path).exists():
        raise FileNotFoundError(f"{context}: path does not exist: {path}")

    inferred = "zarr" if path.endswith(".zarr") else "netcdf"
    file_format = file_format or inferred
    if file_format == "zarr":
        ds = xr.open_zarr(path, **open_kwargs)
    elif file_format in {"netcdf", "nc"}:
        ds = xr.open_dataset(path, **open_kwargs)
    else:
        raise ValueError(f"Unsupported file format {file_format!r} for {path}")

    ds = squeeze_trivial_batch(ds)
    ds = standardize_coords(ds)
    ds = subset_existing_variables(ds, variables, context=context)
    return ds


def normalize_region(region):
    lat0, lat1 = region["latitude"]
    lon0, lon1 = region["longitude"]
    return {"latitude": (float(lat0), float(lat1)), "longitude": (float(lon0), float(lon1))}


def select_latlon_box(da, region):
    """Select a lat/lon region after coordinate normalization, independent of input latitude order."""
    da = standardize_coords(da)
    region = normalize_region(region)
    lat0, lat1 = sorted(region["latitude"])
    lon0, lon1 = sorted(region["longitude"])
    if "latitude" in da.coords:
        da = da.sel(latitude=slice(lat0, lat1))
    if "longitude" in da.coords:
        da = da.sel(longitude=slice(lon0, lon1))
    return da


def ensemble_dims(obj):
    return [d for d in ["init_date", "number"] if d in obj.dims]


## 3. Load CMIP data and compute ACC deltas

The CMIP model set is selected by completeness: a source model must provide every variable in `CONFIG["cmip_required_variables"]`. Optional variables are loaded only when available and do not affect the complete-model intersection.


In [4]:
def build_cmip_manifest(config):
    """Build a manifest with columns source_id, variable_id, path from configured sources."""
    cmip_cfg = config["cmip"]
    rows = []

    explicit = cmip_cfg.get("manifest") or []
    rows.extend(explicit)

    manifest_csv = cmip_cfg.get("manifest_csv")
    if manifest_csv:
        manifest_csv = Path(manifest_csv)
        if manifest_csv.exists():
            rows.extend(pd.read_csv(manifest_csv).to_dict("records"))
        else:
            warnings.warn(f"CMIP manifest_csv does not exist: {manifest_csv}")

    source_ids = cmip_cfg.get("source_ids") or []
    templates = cmip_cfg.get("path_templates") or {}
    for source_id in source_ids:
        for variable_id, template in templates.items():
            rows.append(
                {
                    "source_id": source_id,
                    "variable_id": variable_id,
                    "path": str(template).format(source_id=source_id, variable_id=variable_id),
                }
            )

    if not rows:
        warnings.warn(
            "No CMIP inputs configured. Fill CONFIG['cmip']['manifest'], "
            "CONFIG['cmip']['manifest_csv'], or source_ids/path_templates."
        )
        return pd.DataFrame(columns=["source_id", "variable_id", "path"])

    manifest = pd.DataFrame(rows)
    required_cols = {"source_id", "variable_id", "path"}
    missing_cols = required_cols - set(manifest.columns)
    if missing_cols:
        raise ValueError(f"CMIP manifest is missing columns: {sorted(missing_cols)}")

    manifest["source_id"] = manifest["source_id"].astype(str)
    manifest["variable_id"] = manifest["variable_id"].astype(str)
    manifest["path"] = manifest["path"].astype(str)
    return manifest


def expand_cmip_paths(path_spec):
    """Expand one manifest path entry. Allows concrete paths, globs, and semicolon-separated lists."""
    paths = []
    for part in str(path_spec).split(";"):
        part = part.strip()
        if not part:
            continue
        if any(ch in part for ch in "*?["):
            paths.extend(sorted(glob.glob(part)))
        elif Path(part).exists():
            paths.append(part)
    return sorted(dict.fromkeys(paths))


def cmip_paths_for(manifest, source_id, variable_id):
    rows = manifest[(manifest["source_id"] == source_id) & (manifest["variable_id"] == variable_id)]
    paths = []
    for path_spec in rows["path"]:
        paths.extend(expand_cmip_paths(path_spec))
    return sorted(dict.fromkeys(paths))


def validate_cmip_inputs(manifest, required_variables, optional_variables=None):
    optional_variables = optional_variables or []
    source_ids = sorted(manifest["source_id"].dropna().unique())
    variables = list(dict.fromkeys(list(required_variables) + list(optional_variables)))
    records = []
    for source_id in source_ids:
        for variable_id in variables:
            paths = cmip_paths_for(manifest, source_id, variable_id)
            records.append(
                {
                    "source_id": source_id,
                    "variable_id": variable_id,
                    "n_files": len(paths),
                    "required": variable_id in required_variables,
                    "first_file": paths[0] if paths else None,
                }
            )
    status = pd.DataFrame(records)
    if status.empty:
        warnings.warn("CMIP input status is empty; no source models can be selected.")
        return [], status

    complete = []
    for source_id, group in status.groupby("source_id"):
        have = set(group[(group["required"]) & (group["n_files"] > 0)]["variable_id"])
        missing = sorted(set(required_variables) - have)
        if missing:
            warnings.warn(f"CMIP source {source_id} missing required variables: {missing}")
        else:
            complete.append(source_id)

    if not complete:
        warnings.warn("No CMIP source provides all required variables. ACC delta build will stop until inputs are fixed.")
    return complete, status


def load_cmip_monthly_fields(manifest, source_id, variable_id, config):
    """Open monthly historical CMIP6 fields for one source/variable."""
    paths = cmip_paths_for(manifest, source_id, variable_id)
    if not paths:
        raise FileNotFoundError(f"No CMIP files for source={source_id}, variable={variable_id}")

    cmip_name = config["cmip_variable_map"].get(variable_id)
    if not cmip_name:
        raise KeyError(f"No cmip_variable_map entry for {variable_id!r}")

    ds = xr.open_mfdataset(
        paths,
        combine="by_coords",
        chunks=config["cmip"].get("chunks") or {},
        use_cftime=True,
    )
    ds = standardize_coords(ds)
    if cmip_name not in ds.data_vars:
        raise KeyError(
            f"CMIP variable {cmip_name!r} not found for {source_id}/{variable_id}. "
            "Edit CONFIG['cmip_variable_map'] instead of relying on guessing."
        )
    return standardize_coords(ds[cmip_name])


def compute_cmip_climatology(da, start_year, end_year):
    """Monthly climatology for an inclusive year range."""
    if "time" not in da.dims:
        raise ValueError(f"Expected a time dimension, got dims={da.dims}")
    period = da.sel(time=slice(f"{start_year}-01-01", f"{end_year}-12-31"))
    if period.sizes.get("time", 0) == 0:
        raise ValueError(f"No data in requested period {start_year}-{end_year}")
    clim = period.groupby("time.month").mean("time", skipna=True)
    clim.attrs.update(da.attrs)
    clim.attrs["climatology_period"] = f"{start_year}-{end_year}"
    return clim


def compute_acc_delta(da, modern_period, baseline_period):
    """Paper ACC delta: mean(1980-2014) - mean(1850-1900), by calendar month."""
    modern = compute_cmip_climatology(da, *modern_period)
    baseline = compute_cmip_climatology(da, *baseline_period)
    delta = modern - baseline
    delta.attrs.update(da.attrs)
    delta.attrs["acc_delta_definition"] = (
        f"mean({modern_period[0]}-{modern_period[1]}) - "
        f"mean({baseline_period[0]}-{baseline_period[1]})"
    )
    return delta


def build_cmip_acc_deltas(config):
    """Build per-CMIP-model and multi-model-mean monthly ACC deltas."""
    manifest = build_cmip_manifest(config)
    required = config["cmip_required_variables"]
    optional = config.get("cmip_optional_variables", [])
    complete_sources, status = validate_cmip_inputs(manifest, required, optional)
    display(status)

    if not complete_sources:
        raise RuntimeError("No complete CMIP model set available. Fix the CMIP manifest/config above.")

    variables_to_try = list(dict.fromkeys(required + optional))
    per_source = []
    modern_period = tuple(config["cmip"]["modern_period"])
    baseline_period = tuple(config["cmip"]["baseline_period"])

    for source_id in complete_sources:
        data_vars = {}
        for variable_id in variables_to_try:
            paths = cmip_paths_for(manifest, source_id, variable_id)
            if not paths:
                if variable_id in required:
                    raise FileNotFoundError(f"Required variable disappeared from validated manifest: {source_id}/{variable_id}")
                warnings.warn(f"Optional CMIP variable unavailable for {source_id}: {variable_id}")
                continue
            da = load_cmip_monthly_fields(manifest, source_id, variable_id, config)
            data_vars[variable_id] = compute_acc_delta(da, modern_period, baseline_period)

        ds = xr.Dataset(data_vars).assign_coords(source_id=source_id).expand_dims("source_id")
        per_source.append(ds)

    per_model_delta = xr.concat(per_source, dim="source_id")
    common_delta = per_model_delta.mean("source_id", skipna=True)
    common_delta.attrs.update(
        acc_delta_definition=f"mean({modern_period[0]}-{modern_period[1]}) - mean({baseline_period[0]}-{baseline_period[1]})",
        cmip_sources=", ".join(map(str, complete_sources)),
        note="Shared CMIP-derived ACC perturbation source used for both GenCast and IFS.",
    )
    return per_model_delta, common_delta, status


In [5]:
# Build CMIP-derived ACC perturbation fields.
# This is the heavy CMIP cell. It will warn clearly if inputs are not configured.
per_cmip_model_acc_delta, common_acc_delta_monthly, cmip_input_status = build_cmip_acc_deltas(CONFIG)

# Persist the common perturbation and per-model fields so both forecast systems use the same source.
ensure_parent(CONFIG["outputs"]["cmip_delta_monthly_common"])
common_acc_delta_monthly.to_netcdf(CONFIG["outputs"]["cmip_delta_monthly_common"])

ensure_parent(CONFIG["outputs"]["cmip_delta_by_model"])
per_cmip_model_acc_delta.to_netcdf(CONFIG["outputs"]["cmip_delta_by_model"])

common_acc_delta_monthly


/tmp/user/28413/ipykernel_2845278/753809781.py:30: UserWarning: No CMIP inputs configured. Fill CONFIG['cmip']['manifest'], CONFIG['cmip']['manifest_csv'], or source_ids/path_templates.
  warnings.warn(
/tmp/user/28413/ipykernel_2845278/753809781.py:89: UserWarning: CMIP input status is empty; no source models can be selected.
  warnings.warn("CMIP input status is empty; no source models can be selected.")


""


RuntimeError: No complete CMIP model set available. Fix the CMIP manifest/config above.

## 4. Interpolate ACC deltas

This step interpolates the shared CMIP monthly ACC deltas to each exact initialization date, then to each target forecast grid. Pressure-level variables are interpolated to configured/model-specific pressure levels using log-pressure interpolation by default.


In [ ]:
def _fractional_dayofyear_on_nonleap_calendar(ts):
    ts = pd.Timestamp(ts)
    # Use a non-leap reference year for monthly-climatology interpolation.
    day = min(ts.day, 28) if ts.month == 2 else ts.day
    ref = pd.Timestamp(year=2001, month=ts.month, day=day, hour=ts.hour, minute=ts.minute, second=ts.second)
    frac = (ref.hour / 24.0) + (ref.minute / 1440.0) + (ref.second / 86400.0)
    return float(ref.dayofyear) + frac


def interpolate_delta_to_init_date(monthly_delta, init_date):
    """Linearly interpolate a monthly climatology to the exact initialization date."""
    if "month" not in monthly_delta.dims and "month" not in monthly_delta.coords:
        raise ValueError("ACC delta must have a 'month' dimension/coordinate")

    month_midpoints = np.array([pd.Timestamp(2001, m, 15).dayofyear for m in range(1, 13)], dtype=float)
    base = monthly_delta.assign_coords(month_midpoint=("month", month_midpoints)).swap_dims({"month": "month_midpoint"})
    dec = base.sel(month_midpoint=[month_midpoints[-1]]).assign_coords(month_midpoint=[month_midpoints[-1] - 365.0])
    jan = base.sel(month_midpoint=[month_midpoints[0]]).assign_coords(month_midpoint=[month_midpoints[0] + 365.0])
    extended = xr.concat([dec, base, jan], dim="month_midpoint").sortby("month_midpoint")

    target_doy = _fractional_dayofyear_on_nonleap_calendar(init_date)
    out = extended.interp(month_midpoint=target_doy)
    out = out.drop_vars([v for v in ["month_midpoint", "month"] if v in out.coords], errors="ignore")
    out.attrs.update(monthly_delta.attrs)
    out.attrs["interpolated_to_init_date"] = str(pd.Timestamp(init_date))
    return out


def target_pressure_pa(config, model=None):
    model_cfg = config["forecast_systems"].get(model, {}) if model else {}
    levels_hpa = model_cfg.get("pressure_levels_hpa") or config["target_grid"].get("pressure_levels_hpa")
    if levels_hpa is None:
        return None
    return np.asarray(levels_hpa, dtype=float) * 100.0


def interpolate_pressure(obj, target_pressure, method="linear", coordinate="log"):
    if target_pressure is None or "pressure" not in obj.coords:
        return obj
    target_pressure = np.asarray(target_pressure, dtype=float)
    obj = standardize_coords(obj).sortby("pressure")

    if coordinate == "log":
        source_logp = np.log(np.asarray(obj["pressure"].values, dtype=float))
        target_logp = np.log(target_pressure)
        tmp = obj.assign_coords(log_pressure=("pressure", source_logp)).swap_dims({"pressure": "log_pressure"})
        tmp = tmp.sortby("log_pressure")
        out = tmp.interp(log_pressure=target_logp, method=method)
        out = out.rename({"log_pressure": "pressure"}).assign_coords(pressure=target_pressure)
    else:
        out = obj.interp(pressure=target_pressure, method=method)
    out["pressure"].attrs["units"] = "Pa"
    return out


def infer_target_grid_from_dataset(ds, config, model):
    ds = standardize_coords(ds)
    lat = config["target_grid"].get("latitude")
    lon = config["target_grid"].get("longitude")
    if lat is None:
        if "latitude" not in ds.coords:
            raise ValueError("Could not infer latitude from target dataset; configure CONFIG['target_grid']['latitude']")
        lat = np.asarray(ds["latitude"].values)
    if lon is None:
        if "longitude" not in ds.coords:
            raise ValueError("Could not infer longitude from target dataset; configure CONFIG['target_grid']['longitude']")
        lon = np.asarray(ds["longitude"].values)
    pressure = target_pressure_pa(config, model)
    return {"latitude": np.asarray(lat), "longitude": np.asarray(lon), "pressure": pressure}


def model_variables_for_open(model_cfg, include_diagnostics=True):
    variables = list((model_cfg.get("variable_map") or {}).values())
    if include_diagnostics:
        variables.extend((model_cfg.get("diagnostic_map") or {}).values())
    variables.extend((model_cfg.get("hydrostatic_variables") or {}).values())
    return list(dict.fromkeys([v for v in variables if v]))


def resolve_forecast_paths(config, model, kind, event):
    model_cfg = config["forecast_systems"][model]
    if kind == "factual":
        seed_key = "factual_seed_templates"
        template_key = "factual_template"
    elif kind == "counterfactual":
        seed_key = "counterfactual_seed_templates"
        template_key = "counterfactual_template"
    else:
        raise ValueError("kind must be 'factual' or 'counterfactual'")

    seed_templates = model_cfg.get(seed_key) or []
    seed_paths = [format_template(t, event) for t in seed_templates]
    seed_paths_existing = [p for p in seed_paths if Path(p).exists()]
    if seed_paths and len(seed_paths_existing) == len(seed_paths):
        return seed_paths_existing

    template = model_cfg.get(template_key)
    if template:
        single = format_template(template, event)
        if Path(single).exists():
            return [single]
        warnings.warn(f"{model} {kind} forecast path does not exist: {single}")
    elif seed_paths:
        missing = [p for p in seed_paths if not Path(p).exists()]
        warnings.warn(f"{model} {kind} seed forecast paths missing: {missing}")
    return []


def load_target_grid_for_model_event(model, event, config):
    model_cfg = config["forecast_systems"][model]
    source = model_cfg.get("target_grid_source", "factual_forecast")
    variables = model_variables_for_open(model_cfg, include_diagnostics=True)

    if source == "factual_ic":
        path = format_template(model_cfg["ic_template"], event)
        ds = open_configured_dataset(
            path,
            file_format=model_cfg.get("format"),
            open_kwargs=model_cfg.get("open_kwargs"),
            variables=variables,
            context=f"{model} factual IC target grid",
        )
    else:
        paths = resolve_forecast_paths(config, model, "factual", event)
        if not paths:
            raise FileNotFoundError(f"No factual forecast path available to infer {model} target grid for {event['event_id']}")
        ds = open_configured_dataset(
            paths[0],
            file_format=model_cfg.get("format"),
            open_kwargs=model_cfg.get("open_kwargs"),
            variables=variables,
            context=f"{model} factual forecast target grid",
        )
    return infer_target_grid_from_dataset(ds, config, model)


def interpolate_delta_to_target_grid(init_delta, target_grid, config, model):
    """Interpolate one date-specific ACC delta to a model target grid/levels."""
    out = standardize_coords(init_delta)
    out = interpolate_pressure(
        out,
        target_grid.get("pressure"),
        method=config["target_grid"].get("pressure_method", "linear"),
        coordinate=config["target_grid"].get("pressure_coordinate", "log"),
    )

    lat = np.asarray(target_grid["latitude"], dtype=float)
    lon = np.asarray(target_grid["longitude"], dtype=float)
    lon_norm = ((lon + 180.0) % 360.0) - 180.0
    method = config["target_grid"].get("horizontal_method", "linear")

    if "latitude" in out.coords and "longitude" in out.coords:
        out = out.interp(latitude=lat, longitude=lon_norm, method=method)
    else:
        warnings.warn("ACC delta has no latitude/longitude coordinates; horizontal interpolation skipped.")

    out.attrs.update(init_delta.attrs)
    out.attrs["target_model"] = model
    out.attrs["target_grid_note"] = "Interpolated from shared CMIP ACC delta; same source used for all forecast systems."
    return out


In [ ]:
# Interpolate the shared ACC deltas to each model/event target.
# This assumes `common_acc_delta_monthly` was created in the CMIP cell above.
interpolated_acc_deltas = {}

for model in CONFIG["models"]:
    interpolated_acc_deltas[model] = {}
    for event in CONFIG["events"]:
        init_delta = interpolate_delta_to_init_date(common_acc_delta_monthly, event["init_date"])
        target_grid = load_target_grid_for_model_event(model, event, CONFIG)
        target_delta = interpolate_delta_to_target_grid(init_delta, target_grid, CONFIG, model)
        interpolated_acc_deltas[model][event["event_id"]] = target_delta

        out_path = ensure_parent(format_template(CONFIG["outputs"]["interpolated_delta_template"], event, {"model": model}))
        target_delta.to_netcdf(out_path)
        print(f"saved {model} {event['event_id']} interpolated ACC delta -> {out_path}")

# This dict contains model-grid-specific interpolations of the same shared CMIP perturbation source.
interpolated_acc_deltas.keys()


## 5. Build counterfactual initial conditions

For each model and initialization date:

`counterfactual_ic = factual_ic - ACC_delta_on_model_grid`

Geopotential is handled model-specifically. If `requires_hydrostatic_geopotential=True`, the notebook applies a hydrostatic correction based on the temperature and specific-humidity perturbations rather than blindly subtracting `zg`.


In [ ]:
def load_factual_initial_conditions(model, event, config):
    model_cfg = config["forecast_systems"][model]
    ic_template = model_cfg.get("ic_template")
    if not ic_template:
        raise KeyError(f"No ic_template configured for {model}")
    path = format_template(ic_template, event)
    variables = model_variables_for_open(model_cfg, include_diagnostics=False)
    return open_configured_dataset(
        path,
        file_format="netcdf" if str(path).endswith(".nc") else model_cfg.get("format"),
        open_kwargs={"decode_timedelta": True} if str(path).endswith(".nc") else model_cfg.get("open_kwargs"),
        variables=variables,
        context=f"{model} factual IC {event['event_id']}",
    )


def virtual_temperature(temperature, specific_humidity=None):
    if specific_humidity is None:
        return temperature
    return temperature * (1.0 + 0.609133 * specific_humidity)


def _cumtrapz_hydrostatic(delta_tv_values, pressure_values):
    """Return delta geopotential from top anchor for one vertical profile."""
    y = np.asarray(delta_tv_values, dtype=float)
    p = np.asarray(pressure_values, dtype=float)
    if y.ndim != 1:
        raise ValueError("_cumtrapz_hydrostatic expects a 1-D profile")
    if len(y) != len(p):
        raise ValueError("temperature and pressure profile lengths differ")
    out = np.zeros_like(y, dtype=float)
    if len(y) < 2:
        return out
    lnp = np.log(p)
    for i in range(1, len(y)):
        dlnp = lnp[i] - lnp[i - 1]
        layer_mean = 0.5 * (y[i] + y[i - 1])
        out[i] = out[i - 1] - RD * layer_mean * dlnp
    return out


def hydrostatic_geopotential_delta(delta_tv, pressure_dim="pressure", anchor="top"):
    """Integrate d(delta_phi)/dlnp = -Rd * delta_Tv along pressure."""
    if pressure_dim not in delta_tv.dims:
        raise ValueError(f"{pressure_dim!r} not found in virtual-temperature dims {delta_tv.dims}")

    original_pressure = delta_tv[pressure_dim]
    work = delta_tv.sortby(pressure_dim)  # low pressure to high pressure when Pa is ascending.
    pressure_values = np.asarray(work[pressure_dim].values, dtype=float)

    dphi = xr.apply_ufunc(
        _cumtrapz_hydrostatic,
        work,
        xr.DataArray(pressure_values, dims=[pressure_dim], coords={pressure_dim: pressure_values}),
        input_core_dims=[[pressure_dim], [pressure_dim]],
        output_core_dims=[[pressure_dim]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
    )
    dphi = dphi.assign_coords({pressure_dim: pressure_values})

    if anchor == "surface":
        # Shift so correction is zero at the highest pressure level.
        dphi = dphi - dphi.isel({pressure_dim: -1})
    elif anchor != "top":
        raise ValueError("hydrostatic anchor must be 'top' or 'surface'")

    # Restore original pressure ordering.
    dphi = dphi.interp({pressure_dim: original_pressure})
    dphi.name = "hydrostatic_geopotential_correction"
    dphi.attrs["units"] = "m2 s-2"
    dphi.attrs["definition"] = "counterfactual minus factual hydrostatic geopotential correction"
    return dphi


def hydrostatic_adjust_geopotential(factual_ic, counterfactual_ic, model_cfg):
    if not model_cfg.get("requires_hydrostatic_geopotential", False):
        return counterfactual_ic

    hydro = model_cfg.get("hydrostatic_variables", {})
    t_name = hydro.get("temperature")
    q_name = hydro.get("specific_humidity")
    phi_name = hydro.get("geopotential")

    required = [t_name, phi_name]
    missing = [name for name in required if not name or name not in factual_ic or name not in counterfactual_ic]
    if missing:
        warnings.warn(
            f"Hydrostatic geopotential adjustment skipped; required variables missing: {missing}. "
            "Edit CONFIG['forecast_systems'][model]['hydrostatic_variables'] if names differ."
        )
        return counterfactual_ic

    q_factual = factual_ic[q_name] if q_name and q_name in factual_ic else None
    q_counter = counterfactual_ic[q_name] if q_name and q_name in counterfactual_ic else None

    tv_factual = virtual_temperature(factual_ic[t_name], q_factual)
    tv_counter = virtual_temperature(counterfactual_ic[t_name], q_counter)
    delta_tv = standardize_coords(tv_counter - tv_factual)

    if "pressure" not in delta_tv.dims:
        warnings.warn("Hydrostatic adjustment skipped; temperature perturbation has no pressure dimension.")
        return counterfactual_ic

    dphi = hydrostatic_geopotential_delta(
        delta_tv,
        pressure_dim="pressure",
        anchor=model_cfg.get("hydrostatic_anchor", "top"),
    )

    if model_cfg.get("geopotential_kind", "geopotential") == "height":
        correction = dphi / GRAVITY
        correction.attrs["units"] = "m"
    else:
        correction = dphi

    adjusted = counterfactual_ic.copy()
    adjusted[phi_name] = factual_ic[phi_name] + correction
    adjusted[phi_name].attrs.update(factual_ic[phi_name].attrs)
    adjusted[phi_name].attrs["hydrostatic_adjustment"] = "Applied from temperature/specific-humidity ACC perturbations."
    return adjusted


def build_counterfactual_ic(factual_ic, acc_delta_on_grid, model, config):
    """Subtract ACC deltas from factual IC variables and adjust geopotential if configured."""
    model_cfg = config["forecast_systems"][model]
    variable_map = model_cfg.get("variable_map") or {}
    hydro = model_cfg.get("hydrostatic_variables", {})
    geopotential_var = hydro.get("geopotential")

    factual_ic = standardize_coords(factual_ic)
    acc_delta_on_grid = standardize_coords(acc_delta_on_grid)
    counterfactual = factual_ic.copy(deep=False)

    for cmip_var in acc_delta_on_grid.data_vars:
        target_var = variable_map.get(cmip_var)
        if not target_var:
            warnings.warn(f"No target variable mapping for CMIP variable {cmip_var!r}; skipping.")
            continue
        if target_var not in factual_ic.data_vars:
            warnings.warn(f"{model} factual IC missing target variable {target_var!r} for CMIP {cmip_var!r}; skipping.")
            continue
        if target_var == geopotential_var and model_cfg.get("requires_hydrostatic_geopotential", False):
            # Geopotential is adjusted after thermodynamic perturbations.
            continue

        delta = acc_delta_on_grid[cmip_var]
        try:
            counterfactual[target_var] = factual_ic[target_var] - delta
            counterfactual[target_var].attrs.update(factual_ic[target_var].attrs)
            counterfactual[target_var].attrs["counterfactual"] = f"factual minus CMIP ACC delta variable {cmip_var}"
        except Exception as exc:
            warnings.warn(f"Could not apply ACC delta {cmip_var!r} to {model} variable {target_var!r}: {exc}")

    counterfactual = hydrostatic_adjust_geopotential(factual_ic, counterfactual, model_cfg)
    counterfactual.attrs.update(factual_ic.attrs)
    counterfactual.attrs["counterfactual_definition"] = "factual initial conditions minus shared CMIP ACC perturbation"
    counterfactual.attrs["acc_delta_source"] = config["outputs"]["cmip_delta_monthly_common"]
    return counterfactual


In [ ]:
# Build and save counterfactual ICs for both forecast systems.
# This cell does not run a forecast model; it prepares IC files for the existing generation pipeline.
counterfactual_ic_paths = []

for model in CONFIG["models"]:
    model_cfg = CONFIG["forecast_systems"][model]
    for event in CONFIG["events"]:
        event_id = event["event_id"]
        if model not in interpolated_acc_deltas or event_id not in interpolated_acc_deltas[model]:
            warnings.warn(f"No interpolated ACC delta found for {model}/{event_id}; skipping IC build.")
            continue

        factual_ic = load_factual_initial_conditions(model, event, CONFIG)
        cf_ic = build_counterfactual_ic(factual_ic, interpolated_acc_deltas[model][event_id], model, CONFIG)

        out_path = ensure_parent(format_template(model_cfg["counterfactual_ic_template"], event))
        cf_ic.to_netcdf(out_path)
        counterfactual_ic_paths.append({"model": model, "event_id": event_id, "path": str(out_path)})
        print(f"saved {model} {event_id} counterfactual IC -> {out_path}")

pd.DataFrame(counterfactual_ic_paths)


## 6. Forecast handling

This notebook expects factual and counterfactual forecast outputs to use the same initialization dates, lead times, and lagged/seed ensemble structure. It does not alter the forecast-generation pipeline. If counterfactual forecasts are not present yet, run your existing forecast pipeline using the counterfactual ICs saved above, then rerun this section.


In [ ]:
def add_valid_time(ds, init_date, time_is_relative_to_init):
    ds = standardize_coords(ds)
    init64 = np.datetime64(pd.Timestamp(init_date).to_datetime64())

    if time_is_relative_to_init:
        if "time" not in ds.coords and "time" not in ds.dims:
            warnings.warn("Relative-time forecast has no time coordinate; valid_time not added.")
            return ds
        time_values = ds["time"].values
        valid_time = init64 + time_values
        if "time" in ds.dims:
            ds = ds.assign_coords(lead_time=("time", time_values), valid_time=("time", valid_time))
            ds = ds.swap_dims({"time": "valid_time"})
        else:
            ds = ds.assign_coords(valid_time=valid_time)
    else:
        if "valid_time" not in ds.dims and "time" in ds.dims:
            ds = ds.rename({"time": "valid_time"})
        elif "valid_time" not in ds.coords and "time" in ds.coords:
            ds = ds.assign_coords(valid_time=ds["time"])

    return ds


def select_event_time_window(ds, event, config):
    if "valid_time" not in ds.coords and "valid_time" not in ds.dims:
        return ds
    start, end = event.get("event_window", (None, None))
    if start is not None or end is not None:
        return ds.sel(valid_time=slice(start, end))

    lead_start, lead_end = config.get("lead_time_window_hours", (None, None))
    if lead_start is None and lead_end is None:
        return ds
    init = pd.Timestamp(event["init_date"])
    abs_start = init + pd.Timedelta(hours=lead_start or 0)
    abs_end = None if lead_end is None else init + pd.Timedelta(hours=lead_end)
    return ds.sel(valid_time=slice(abs_start, abs_end))


def combine_seed_ensembles(paths, model_cfg, variables, init_date, context):
    datasets = []
    offset = 0
    for path in paths:
        ds = open_configured_dataset(
            path,
            file_format=model_cfg.get("format"),
            open_kwargs=model_cfg.get("open_kwargs"),
            variables=variables,
            context=context,
        )
        ds = add_valid_time(ds, init_date, model_cfg.get("time_is_relative_to_init", False))
        ds = standardize_coords(ds)

        if "number" in ds.dims:
            nmem = ds.sizes["number"]
            ds = ds.assign_coords(number=np.arange(offset, offset + nmem))
            offset += nmem
        datasets.append(ds)

    if len(datasets) == 1:
        return datasets[0]
    return xr.concat(datasets, dim="number").sortby("number")


def load_forecast_for_event(model, kind, event, config):
    model_cfg = config["forecast_systems"][model]
    diagnostic_vars = list((model_cfg.get("diagnostic_map") or {}).values())
    paths = resolve_forecast_paths(config, model, kind, event)
    if not paths:
        warnings.warn(f"No {kind} forecast available for {model}/{event['event_id']}")
        return None

    ds = combine_seed_ensembles(
        paths,
        model_cfg=model_cfg,
        variables=diagnostic_vars,
        init_date=event["init_date"],
        context=f"{model} {kind} forecast {event['event_id']}",
    )
    ds = select_event_time_window(ds, event, config)
    ds = ds.expand_dims(init_date=[pd.Timestamp(event["init_date"])])
    ds.attrs["forecast_kind"] = kind
    ds.attrs["forecast_model"] = model
    ds.attrs["event_id"] = event["event_id"]
    return ds


def load_lagged_ensemble(model, kind, events, config):
    datasets = []
    for event in events:
        ds = load_forecast_for_event(model, kind, event, config)
        if ds is not None:
            datasets.append(ds)
    if not datasets:
        warnings.warn(f"No {kind} lagged ensemble datasets loaded for {model}")
        return None
    return xr.concat(datasets, dim="init_date").sortby("init_date")


In [ ]:
forecast_inputs = {}

for model in CONFIG["models"]:
    factual = load_lagged_ensemble(model, "factual", CONFIG["events"], CONFIG)
    counterfactual = load_lagged_ensemble(model, "counterfactual", CONFIG["events"], CONFIG)
    forecast_inputs[model] = {"factual": factual, "counterfactual": counterfactual}

forecast_inputs


## 7. Diagnostics

ACC signal is computed as:

`ACC_signal = factual lagged-ensemble mean - counterfactual lagged-ensemble mean`

The comparison preserves lagged initialization dates and member dimensions until the final ensemble mean/spread reduction.


In [ ]:
def daily_mean_diagnostics(ds):
    if ds is None:
        return None
    if "valid_time" in ds.dims:
        return ds.resample(valid_time="1D").mean()
    return ds


def diagnostic_variables_present(model, ds, config):
    if ds is None:
        return []
    model_map = config["forecast_systems"][model].get("diagnostic_map") or {}
    out = []
    for diag_name, var_name in model_map.items():
        if var_name in ds.data_vars:
            out.append((diag_name, var_name))
    return out


def compare_factual_vs_counterfactual(factual, counterfactual, model, config):
    if factual is None or counterfactual is None:
        warnings.warn(f"Cannot compare {model}; factual or counterfactual forecast is missing.")
        return None

    factual = daily_mean_diagnostics(standardize_coords(factual))
    counterfactual = daily_mean_diagnostics(standardize_coords(counterfactual))
    factual, counterfactual = xr.align(factual, counterfactual, join="inner")

    member_dims = [d for d in ["number"] if d in factual.dims]
    lagged_dims = [d for d in ["init_date", "number"] if d in factual.dims]
    if not lagged_dims:
        warnings.warn(f"{model}: no lagged/member ensemble dimensions found; computing deterministic difference.")

    result = xr.Dataset()
    for diag_name, var_name in diagnostic_variables_present(model, factual, config):
        if var_name not in counterfactual:
            warnings.warn(f"{model}: counterfactual missing diagnostic variable {var_name!r}")
            continue

        # Preserve init-date structure for per-init checks.
        factual_by_init = factual[var_name].mean(member_dims, skipna=True) if member_dims else factual[var_name]
        counter_by_init = counterfactual[var_name].mean(member_dims, skipna=True) if member_dims else counterfactual[var_name]
        signal_by_init = factual_by_init - counter_by_init

        result[f"{diag_name}_factual_mean_by_init"] = factual_by_init
        result[f"{diag_name}_counterfactual_mean_by_init"] = counter_by_init
        result[f"{diag_name}_acc_signal_by_init"] = signal_by_init

        # Paper-style lagged-ensemble mean across init dates and members.
        factual_mean = factual[var_name].mean(lagged_dims, skipna=True) if lagged_dims else factual[var_name]
        counter_mean = counterfactual[var_name].mean(lagged_dims, skipna=True) if lagged_dims else counterfactual[var_name]
        signal = factual_mean - counter_mean

        result[f"{diag_name}_factual_mean"] = factual_mean
        result[f"{diag_name}_counterfactual_mean"] = counter_mean
        result[f"{diag_name}_acc_signal"] = signal

        if lagged_dims:
            result[f"{diag_name}_factual_spread"] = factual[var_name].std(lagged_dims, skipna=True)
            result[f"{diag_name}_counterfactual_spread"] = counterfactual[var_name].std(lagged_dims, skipna=True)

    result.attrs.update(
        forecast_model=model,
        acc_signal_definition="factual lagged-ensemble mean minus counterfactual lagged-ensemble mean",
        acc_delta_source=config["outputs"]["cmip_delta_monthly_common"],
    )
    return result

def reduce_event_summary(da, event, reducer="mean_then_max_time"):
    work = select_latlon_box(da, event["summary_region"])
    spatial_dims = [d for d in ["latitude", "longitude"] if d in work.dims]
    if spatial_dims:
        work = work.mean(spatial_dims, skipna=True)

    if reducer == "mean_then_max_time":
        if "valid_time" in work.dims:
            work = work.max("valid_time", skipna=True)
    elif reducer == "mean_time":
        if "valid_time" in work.dims:
            work = work.mean("valid_time", skipna=True)
    elif reducer == "max_space_time":
        dims = [d for d in ["valid_time", "latitude", "longitude"] if d in da.dims]
        work = da.max(dims, skipna=True) if dims else da
    else:
        raise ValueError(f"Unsupported reducer: {reducer}")

    return float(work.values) if np.ndim(work.values) == 0 else work


def build_summary_table(results_by_model, events, config):
    rows = []
    for model, result in results_by_model.items():
        if result is None:
            continue
        for diag_name, diag_cfg in config["diagnostics"].items():
            lagged_signal_name = f"{diag_name}_acc_signal"
            lagged_factual_name = f"{diag_name}_factual_mean"
            lagged_counter_name = f"{diag_name}_counterfactual_mean"
            if lagged_signal_name in result:
                # One paper-style lagged-ensemble summary over all configured init dates.
                first_event = events[0]
                reducer = diag_cfg.get("summary_reducer", "mean_then_max_time")
                rows.append(
                    {
                        "model": model,
                        "event_id": "lagged_ensemble_all_configured_inits",
                        "init_date": ";".join(event["init_date"] for event in events),
                        "diagnostic": diag_name,
                        "ensemble_scope": "lagged_init_dates_and_members",
                        "factual_summary": reduce_event_summary(result[lagged_factual_name], first_event, reducer) if lagged_factual_name in result else np.nan,
                        "counterfactual_summary": reduce_event_summary(result[lagged_counter_name], first_event, reducer) if lagged_counter_name in result else np.nan,
                        "acc_signal_summary": reduce_event_summary(result[lagged_signal_name], first_event, reducer),
                        "units": diag_cfg.get("units", ""),
                        "reducer": reducer,
                    }
                )

            by_init_signal_name = f"{diag_name}_acc_signal_by_init"
            by_init_factual_name = f"{diag_name}_factual_mean_by_init"
            by_init_counter_name = f"{diag_name}_counterfactual_mean_by_init"
            if by_init_signal_name not in result:
                continue
            reducer = diag_cfg.get("summary_reducer", "mean_then_max_time")
            for event in events:
                event_result = result
                if "init_date" in event_result[by_init_signal_name].coords:
                    event_result = event_result.sel(init_date=pd.Timestamp(event["init_date"]), method="nearest")
                rows.append(
                    {
                        "model": model,
                        "event_id": event["event_id"],
                        "init_date": event["init_date"],
                        "diagnostic": diag_name,
                        "ensemble_scope": "members_with_init_date_preserved",
                        "factual_summary": reduce_event_summary(event_result[by_init_factual_name], event, reducer) if by_init_factual_name in event_result else np.nan,
                        "counterfactual_summary": reduce_event_summary(event_result[by_init_counter_name], event, reducer) if by_init_counter_name in event_result else np.nan,
                        "acc_signal_summary": reduce_event_summary(event_result[by_init_signal_name], event, reducer),
                        "units": diag_cfg.get("units", ""),
                        "reducer": reducer,
                    }
                )
    return pd.DataFrame(rows)


In [ ]:
acc_results = {}

for model in CONFIG["models"]:
    acc_results[model] = compare_factual_vs_counterfactual(
        forecast_inputs[model]["factual"],
        forecast_inputs[model]["counterfactual"],
        model,
        CONFIG,
    )

summary_table = build_summary_table(acc_results, CONFIG["events"], CONFIG)
summary_table


## Plots

These plots compare GenCast and IFS ACC signals using the same CMIP-derived perturbation source. They are intentionally simple so event-region and time-evolution behavior is easy to inspect.


In [ ]:
def plot_event_time_evolution(summary_source, results_by_model, events, diagnostic="t2m"):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for model, result in results_by_model.items():
        if result is None:
            continue
        signal_name = f"{diagnostic}_acc_signal"
        if signal_name not in result:
            continue

        for event in events:
            da = result[signal_name]
            if "init_date" in da.coords:
                da = da.sel(init_date=pd.Timestamp(event["init_date"]), method="nearest")
            region_da = select_latlon_box(da, event["summary_region"])
            spatial_dims = [d for d in ["latitude", "longitude"] if d in region_da.dims]
            if spatial_dims:
                region_da = region_da.mean(spatial_dims, skipna=True)
            if "valid_time" in region_da.dims:
                ax.plot(region_da["valid_time"].values, region_da.values, marker="o", label=f"{model} {event['init_date'][:10]}")
            else:
                ax.scatter(pd.Timestamp(event["init_date"]), float(region_da), label=f"{model} {event['init_date'][:10]}")

    ax.axhline(0, color="k", linewidth=0.8)
    ax.set_title(f"ACC signal time evolution: {diagnostic}")
    ax.set_ylabel(CONFIG["diagnostics"].get(diagnostic, {}).get("units", ""))
    ax.legend(loc="best", fontsize=8)
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig, ax


def plot_signal_map(result, event, diagnostic="t2m", model="model"):
    signal_name = f"{diagnostic}_acc_signal"
    if result is None or signal_name not in result:
        warnings.warn(f"No signal map available for {model}/{diagnostic}")
        return None, None

    da = result[signal_name]
    if "init_date" in da.coords:
        da = da.sel(init_date=pd.Timestamp(event["init_date"]), method="nearest")
    if "valid_time" in da.dims:
        da = da.mean("valid_time", skipna=True)

    if HAS_CARTOPY and {"latitude", "longitude"}.issubset(set(da.coords)):
        fig = plt.figure(figsize=(7, 4.5))
        ax = plt.axes(projection=ccrs.PlateCarree())
        im = da.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="RdBu_r", robust=True, add_colorbar=True)
        ax.coastlines(linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, linewidth=0.3)
        region = normalize_region(event["summary_region"])
        ax.set_extent([region["longitude"][0] - 10, region["longitude"][1] + 10, region["latitude"][0] - 8, region["latitude"][1] + 8])
    else:
        fig, ax = plt.subplots(figsize=(7, 4.5))
        im = da.plot(ax=ax, cmap="RdBu_r", robust=True, add_colorbar=True)

    ax.set_title(f"{model} {event['event_id']} {diagnostic} ACC signal")
    fig.tight_layout()
    return fig, ax


In [ ]:
plot_dir = Path(CONFIG["outputs"]["plot_dir"])
plot_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plot_event_time_evolution(summary_table, acc_results, CONFIG["events"], diagnostic="t2m")
fig.savefig(plot_dir / "t2m_acc_signal_time_evolution_gencast_vs_ifs.png", dpi=150)
plt.show()

for model, result in acc_results.items():
    for event in CONFIG["events"]:
        fig, ax = plot_signal_map(result, event, diagnostic="t2m", model=model)
        if fig is not None:
            fig.savefig(plot_dir / f"{model}_{event['event_id']}_t2m_acc_signal_map.png", dpi=150)
            plt.show()


## 8. Outputs

This section saves compact outputs for downstream comparison:

- CMIP-derived common monthly ACC deltas
- Per-CMIP-model ACC deltas
- Per-model/event interpolated perturbations
- Per-model ACC signal fields
- GenCast-vs-IFS comparison plots
- Event-level summary table


In [ ]:
# Save per-model attribution fields and compact event summary.
for model, result in acc_results.items():
    if result is None:
        continue
    # Save one combined model result; event-specific filenames are available via the template if preferred.
    out_path = ensure_parent(OUTPUT_DIR / "results" / model / f"{model}_all_events_acc_signal.nc")
    result.to_netcdf(out_path)
    print(f"saved {model} ACC result -> {out_path}")

summary_csv = ensure_parent(CONFIG["outputs"]["summary_csv"])
summary_table.to_csv(summary_csv, index=False)
print(f"saved summary table -> {summary_csv}")

summary_table


## Notes and checks before interpreting results

- The same `common_acc_delta_monthly` object is the perturbation source for both GenCast and IFS. Model-specific interpolation changes only the grid/levels, not the CMIP model set or ACC definition.
- Missing CMIP variables are warnings in the validation table and become hard failures if no complete CMIP model set remains.
- Counterfactual forecasts must be generated externally from the counterfactual ICs unless your existing pipeline writes them before Section 6 is run.
- Hydrostatic geopotential adjustment is intentionally model-specific through `CONFIG["forecast_systems"][model]["hydrostatic_variables"]`.
